# Molmo2-SGCoT Training

## 0. Configuration

Set all your training hyperparameters here in one place.

In [ ]:
# ============================================================================
# CONFIGURATION — Edit these to fit your needs
# ============================================================================

# Model
MODEL_ID = "allenai/Molmo2-8B"

# Data
DATA_PATH = "/content/train.jsonl"
MAX_LENGTH = 16000

# Training mode: "lora" or "qlora"
TRAINING_MODE = "qlora"

# LoRA hyperparameters
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0
# Target modules in the Molmo2 LLM (Qwen3-8B backbone):
#   att_proj  = fused QKV projection
#   attn_out  = attention output projection
#   ff_proj   = MLP gate+up projection (fused)
#   ff_out    = MLP down projection
LORA_TARGET_MODULES = ["att_proj", "attn_out", "ff_proj", "ff_out"]

# Training hyperparameters
OUTPUT_DIR = "/content/checkpoints/molmo2-finetune"
NUM_EPOCHS = 1
BATCH_SIZE = 64
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01
LR_SCHEDULER = "cosine"
LOGGING_STEPS = 1
SAVE_STEPS = 100
SAVE_TOTAL_LIMIT = 2

PUSH_TO_HUB = False
HUB_MODEL_ID = ""

print(f"Training mode: {TRAINING_MODE}")
print(f"Model: {MODEL_ID}")
print(f"Output: {OUTPUT_DIR}")

Training mode: qlora
Model: allenai/Molmo2-8B
Output: /content/checkpoints/molmo2-finetune


Prepare dataset for training. We use the following 300 sample data to train Molmo2-SGCoT:

In [ ]:
!curl -L -o train.jsonl "https://huggingface.co/datasets/tiedong/Molmo2-SGCoT/resolve/main/train.jsonl"

## 1. Install Dependencies

In [ ]:
%%capture
!pip install transformers==4.57.1
!pip install torch pillow einops torchvision accelerate
!pip install peft bitsandbytes
!pip install molmo_utils
!pip install "molmo_utils[torchcodec]"
!pip install decord2

# Optional: for logging to Weights & Biases
# !pip install wandb

## 2. Load Processor & Model

In [ ]:
import torch, logging
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

logging.basicConfig(format="%(asctime)s %(levelname)s %(message)s", level=logging.INFO)

print("Loading processor...")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

print(f"Loading model ({TRAINING_MODE})...")
model_kwargs = dict(
    pretrained_model_name_or_path=MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
if TRAINING_MODE == "qlora":
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
        # Do NOT quantize the vision backbone.
        # The backbone's .dtype property returns its weight dtype, which becomes
        # uint8 when quantized. It then casts pixel_values to uint8 internally,
        # crashing nn.LayerNorm with "not implemented for 'Byte'".
        llm_int8_skip_modules=["vision_backbone"],
    )

model = AutoModelForImageTextToText.from_pretrained(**model_kwargs)
print(f"Model loaded. dtype={model.dtype}")


Loading processor...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading model (qlora)...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

Model loaded. dtype=torch.bfloat16


## 3. Freeze Vision Encoder & Apply LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# Freeze SigLIP 2 vision encoder
for param in model.model.vision_backbone.image_vit.parameters():
    param.requires_grad = False
print(f"Frozen ViT params: {sum(p.numel() for p in model.model.vision_backbone.image_vit.parameters()):,}")

if TRAINING_MODE == "qlora":
    model = prepare_model_for_kbit_training(
        model, use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    )
else:
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )

model = get_peft_model(model, LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES, bias="none", task_type=TaskType.CAUSAL_LM,
))
model.print_trainable_parameters()

Frozen ViT params: 382,505,936
trainable params: 36,569,088 || all params: 8,698,272,208 || trainable%: 0.4204


## 4. Dataset & Data Collator

In [ ]:
import os, json, torch
from torch.utils.data import Dataset
from molmo_utils import process_vision_info


def mask_trajectory_only(input_ids, labels, tokenizer):
    """
    Mask only the trajectory: from the opening " (after coords=) to </tracks>.
    Leaves <tracks coords= unmasked so the model sees the tag start.
    Works on batched [1, seq_len] tensors.
    """
    # prefix = "<tracks coords=\"0.0 1"
    prefix = ""
    suffix = "</tracks>"
    prefix_tokens = tokenizer.encode(prefix, add_special_tokens=False)
    suffix_tokens = tokenizer.encode(suffix, add_special_tokens=False)

    ids = input_ids[0].tolist()
    start_mask = None  # first token to mask (inclusive)
    end_mask = None    # one past last token to mask

    # Find end of "<tracks coords=" -> trajectory starts at next token
    for i in range(len(ids) - len(prefix_tokens) + 1):
        if ids[i : i + len(prefix_tokens)] == prefix_tokens:
            start_mask = i + len(prefix_tokens)
            break

    # Find </tracks> (mask through end of this tag)
    for i in range(len(ids) - len(suffix_tokens) + 1):
        if ids[i : i + len(suffix_tokens)] == suffix_tokens:
            end_mask = i + len(suffix_tokens)
            break

    if start_mask is not None and end_mask is not None and start_mask < end_mask:
        labels[0, start_mask:end_mask] = -100

    return labels



def _process_messages(processor, messages, add_generation_prompt, max_length=None, truncation=False):
    """
    Two-step processing that respects max_fps / num_frames in the message dict.
    1) process_vision_info() loads videos/images using the params in the messages
    2) processor() tokenizes text + visual features
    """
    # Step 1: Get text template (no tokenization yet)
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=add_generation_prompt,
    )

    # Step 2: Extract & load visual inputs (respects max_fps, num_frames, etc.)
    images, videos, video_kwargs = process_vision_info(messages)
    video_metadatas = None
    if videos is not None:
        videos, video_metadatas = zip(*videos)
        videos = list(videos)
        video_metadatas = list(video_metadatas)

    # Step 3: Process everything through the processor
    proc_kwargs = dict(
        text=text,
        images=images,
        videos=videos,
        video_metadata=video_metadatas,
        return_tensors="pt",
        **video_kwargs,
    )
    if max_length is not None:
        proc_kwargs["max_length"] = max_length
        proc_kwargs["truncation"] = truncation

    return processor(**proc_kwargs)


class Molmo2VideoDataset(Dataset):
    """
    Returns processor output *exactly as-is* (with batch dim from return_tensors='pt'),
    plus `labels` for next-token prediction loss.

    Uses process_vision_info() so that max_fps, num_frames, etc. in the
    message dict are respected.
    """
    def __init__(self, data_path, processor, max_length=8192):
        self.processor = processor
        self.max_length = max_length
        self.data = []
        with open(data_path) as f:
            for line in f:
                line = line.strip()
                if line:
                    self.data.append(json.loads(line))
        print(f"Loaded {len(self.data)} examples from {data_path}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        messages = self.data[idx]["messages"]

        # --- Full conversation (including assistant) for loss ---
        full = _process_messages(
            self.processor, messages,
            add_generation_prompt=False,
            max_length=self.max_length, truncation=True,
        )
        full_len = full["input_ids"].shape[1]

        # --- Prompt-only prefix to find where assistant starts ---
        user_prefix = []
        for m in messages:
            if m["role"] != "assistant":
                user_prefix.append(m)
            else:
                break

        prompt = _process_messages(
            self.processor, user_prefix,
            add_generation_prompt=True,
            max_length=self.max_length, truncation=True,
        )
        prompt_len = min(prompt["input_ids"].shape[1], full_len)

        # --- Labels: mask prompt with -100, keep assistant tokens ---
        labels = full["input_ids"].clone()        # [1, seq_len]
        labels[0, :prompt_len] = -100


        # 2️⃣ Mask only trajectory (from " after coords= to </tracks>)
        labels = mask_trajectory_only(
            full["input_ids"],
            labels,
            self.processor.tokenizer
        )


        pad_id = self.processor.tokenizer.pad_token_id
        if pad_id is not None:
            labels[labels == pad_id] = -100
        full["labels"] = labels

        return full                                # dict of tensors, all with batch dim = 1


def passthrough_collator(batch):
    """
    batch_size=1 passthrough. Returns the single item's dict as-is.
    Every tensor already has its batch dimension from the processor.
    """
    assert len(batch) == 1, (
        f"This collator only supports batch_size=1, got {len(batch)}. "
        "Use gradient_accumulation_steps for larger effective batches."
    )
    return batch[0]


# Build dataset
train_dataset = Molmo2VideoDataset(DATA_PATH, processor, MAX_LENGTH)

# Sanity check: load one sample, print shapes
print("\nSanity check — sample 0:")
sample = train_dataset[0]
for k, v in sample.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k:30s} {str(v.shape):20s} {v.dtype}")
    else:
        print(f"  {k:30s} {type(v).__name__}")
n_train = (sample["labels"] != -100).sum().item()
total = sample["labels"].shape[1]
print(f"\n  Token summary:")
print(f"    input_ids length:   {total}")
print(f"    prompt tokens:      {total - n_train}")
print(f"    trainable tokens:   {n_train}")
if "pixel_values_videos" in sample:
    n_frames = sample["pixel_values_videos"].shape[0]
    print(f"    video frames:       {n_frames}")
    print(f"    (if max_fps=2 and video is ~{n_frames/2:.0f}s, or num_frames={n_frames})")
elif "pixel_values" in sample:
    print(f"    image crops:        {sample['pixel_values'].shape[0]}")
if "video_token_pooling" in sample:
    print(f"    video pooled patches: {sample['video_token_pooling'].shape[0]}")
if n_train == 0:
    print(f"\n  *** WARNING: 0 trainable tokens! The assistant response was likely")
    print(f"  *** truncated away. Increase MAX_LENGTH (currently {MAX_LENGTH}).")
    print(f"  *** Or reduce num_frames / max_fps in your video entries.")



# Debug: print masked and trained token spans for first datapoint
def _spans_from_mask(mask):
    indices = torch.where(mask)[0]
    if len(indices) == 0:
        return []
    spans = []
    i = 0
    while i < len(indices):
        start = indices[i].item()
        while i + 1 < len(indices) and indices[i + 1].item() == indices[i].item() + 1:
            i += 1
        end = indices[i].item() + 1
        spans.append((start, end))
        i += 1
    return spans

tokenizer = train_dataset.processor.tokenizer
input_ids = sample["input_ids"][0]
labels = sample["labels"][0]
masked_spans = _spans_from_mask(labels == -100)
trained_spans = _spans_from_mask(labels != -100)

print(f"\n  Masked token spans (first datapoint, {len(masked_spans)} span(s)):")
for start, end in masked_spans:
    decoded = tokenizer.decode(input_ids[start:end].tolist(), skip_special_tokens=False)
    print(f"    [{start}:{end}] {decoded!r}")

print(f"\n  Trained token spans (first datapoint, {len(trained_spans)} span(s)):")
for start, end in trained_spans:
    decoded = tokenizer.decode(input_ids[start:end].tolist(), skip_special_tokens=False)
    print(f"    [{start}:{end}] {decoded!r}")

Loaded 300 examples from /content/train.jsonl

Sanity check — sample 0:
  input_ids                      torch.Size([1, 403]) torch.int64
  attention_mask                 torch.Size([1, 403]) torch.int64
  token_type_ids                 torch.Size([1, 403]) torch.bool
  labels                         torch.Size([1, 403]) torch.int64

  Token summary:
    input_ids length:   403
    prompt tokens:      399
    trainable tokens:   4

  Masked token spans (first datapoint, 1 span(s)):
    [0:399] '<|im_end|><|im_start|>user\nTrack the cup that contains the ball and answer which cup contains the ball at the end of the video.<|im_end|>\n <|im_start|>assistant\n<tracks coords="0.0 1 772 524;0.5 1 805 310;1.0 1 800 310;1.5 1 790 403;2.0 1 772 536;2.5 1 686 767;3.0 1 444 825;3.5 1 251 683;4.0 1 220 511;4.5 1 354 341;5.0 1 543 290;5.5 1 705 374;6.0 1 775 547;6.5 1 739 672;7.0 1 626 705;7.5 1 525 633;8.0 1 498 559;8.5 1 449 395;9.0 1 351 369;9.5 1 264 435;10.0 1 213 524;10.5 1 217 519;11.0 1 218

## 5. Train

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=1,              # MUST be 1 (passthrough collator)
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    tf32=True,
    dataloader_num_workers=0,                   # 0 avoids multiprocess issues with video decoding
    remove_unused_columns=False,                # CRITICAL for Molmo2
    report_to="none",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_grad_norm=1.0,
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=passthrough_collator,
)

eff_bs = 1 * GRADIENT_ACCUMULATION_STEPS
steps = len(train_dataset) * NUM_EPOCHS // eff_bs
print(f"Dataset: {len(train_dataset)} samples, {NUM_EPOCHS} epochs")
print(f"Effective batch size: {eff_bs}, ~{steps} optimization steps")

Dataset: 300 samples, 1 epochs
Effective batch size: 4, ~75 optimization steps


In [ ]:
# Start training!
train_result = trainer.train()

print("\n" + "="*60)
print("Training complete!")
print("="*60)
for k, v in train_result.metrics.items():
    print(f"  {k}: {v}")

Step,Training Loss
1,7.537600
2,6.938600
3,6.674200
4,6.124500
5,5.187200
6,3.782400
7,3.190400
8,2.150300
9,3.036600
10,1.240700



Training complete!
  train_runtime: 186.0468
  train_samples_per_second: 1.612
  train_steps_per_second: 0.403
  total_flos: 6294069769708800.0
  train_loss: 0.692133291677698
  epoch: 1.0


## 6. Save Model

In [ ]:
trainer.save_model(OUTPUT_DIR)
processor.tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to {OUTPUT_DIR}")
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(f"  {f:40s} {os.path.getsize(os.path.join(OUTPUT_DIR, f))/1e6:.1f} MB")

Saved to /content/checkpoints/molmo2-finetune
  README.md                                0.0 MB
  adapter_config.json                      0.0 MB
  adapter_model.safetensors                146.3 MB
  added_tokens.json                        0.0 MB
  chat_template.jinja                      0.0 MB
  checkpoint-100                           0.0 MB
  checkpoint-75                            0.0 MB
  merges.txt                               1.7 MB
  special_tokens_map.json                  0.0 MB
  tokenizer.json                           11.5 MB
  tokenizer_config.json                    0.1 MB
  training_args.bin                        0.0 MB
  vocab.json                               2.8 MB


## 7. Inference

Download VET-Bench videos

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="tiedong/vetbench",
    repo_type="dataset",
    local_dir="vetbench"
)

In [ ]:
import os
import json
import gc
import re
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import PeftModel
from molmo_utils import process_vision_info

# left->A, middle->B, right->C
LETTER_TO_POSITION = {"A": "Left", "B": "Middle", "C": "Right"}
POSITION_TO_LETTER = {"left": "A", "middle": "B", "right": "C"}


def extract_predicted(response: str) -> str | None:
    """Extract A/B/C from response: look for word 'left' -> A, 'middle' -> B, 'right' -> C."""
    lower = response.lower()
    for pos, let in POSITION_TO_LETTER.items():
        if re.search(rf"\b{re.escape(pos)}\b", lower):
            return let
    return None

BASE_MODEL = "allenai/Molmo2-8B"
LORA_CHECKPOINT = "tiedong/Molmo2-SGCoT"
# LORA_CHECKPOINT = "checkpoints/molmo2-finetune/"
MODEL_NAME = "Molmo2-8B-finetuned"
MAX_FPS = 8
MAX_NEW_TOKENS = 1024
TASKS = ["card", "cup"]

# change accordingly
BASE_DIR = "/content/vetbench"

TASK_PROMPTS = {
    "cup": "Track the cup that contains the ball and answer which cup contains the ball at the end of the video.",
    "card": "Track the Queen of Hearts and answer where is the Queen of Hearts at the end of the video.",
}

# Load processor from base model (checkpoint-50 only has adapter weights, no processor)
processor = AutoProcessor.from_pretrained(BASE_MODEL, trust_remote_code=True)

# Load base model then merge LoRA weights
model = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    torch_dtype="auto",
    device_map="auto",
)
model = PeftModel.from_pretrained(model, LORA_CHECKPOINT)
model.eval()

im_end_id = processor.tokenizer.convert_tokens_to_ids("<|im_end|>")

def _process_messages(processor, messages, add_generation_prompt, max_length=None, truncation=False):
    """
    Two-step processing that respects max_fps / num_frames in the message dict.
    1) process_vision_info() loads videos/images using the params in the messages
    2) processor() tokenizes text + visual features
    """
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=add_generation_prompt,
    )

    images, videos, video_kwargs = process_vision_info(messages)
    video_metadatas = None
    if videos is not None:
        videos, video_metadatas = zip(*videos)
        videos = list(videos)
        video_metadatas = list(video_metadatas)

    proc_kwargs = dict(
        text=text,
        images=images,
        videos=videos,
        video_metadata=video_metadatas,
        return_tensors="pt",
        **video_kwargs,
    )
    if max_length is not None:
        proc_kwargs["max_length"] = max_length
        proc_kwargs["truncation"] = truncation

    return processor(**proc_kwargs)


def analyze_video(video_path, prompt_text):
    print(f"--- Processing {video_path} ---")
    if not os.path.exists(video_path):
        raise FileNotFoundError(f"Video not found: {video_path}")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt_text},
                {"type": "video", "video": video_path, "max_fps": MAX_FPS},
            ],
        }
    ]

    inputs = _process_messages(processor, messages, add_generation_prompt=True)
    inputs = {
        k: v.to(model.device) if isinstance(v, torch.Tensor) else v
        for k, v in inputs.items()
    }
    prompt_len = inputs["input_ids"].shape[1]

    try:
        with torch.inference_mode():
            gen_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                eos_token_id=im_end_id,
            )
        response = processor.tokenizer.decode(
            gen_ids[0, prompt_len:], skip_special_tokens=False,
        )
        return response.strip()
    finally:
        gc.collect()
        torch.cuda.empty_cache()


def save_results(results, output_path):
    keys_to_drop = {"task", "task_type", "ground_truth"}
    to_save = [{k: v for k, v in r.items() if k not in keys_to_drop} for r in results]
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(to_save, f, indent=2, ensure_ascii=False)


for task in TASKS:
    qa_path = os.path.join(BASE_DIR, task, "QA.json")
    video_dir = os.path.join(BASE_DIR, task)
    output_path = os.path.join(BASE_DIR, "output", task, f"{MODEL_NAME}.json")
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    with open(qa_path, "r", encoding="utf-8") as f:
        qa_data = json.load(f)

    results = []
    correct_so_far = 0

    prompt = TASK_PROMPTS[task]
    for i, item in enumerate(qa_data):
        video_name = item["video"]
        video_path = os.path.join(video_dir, video_name)

        print(f"\n{'='*50}")
        print(f"[{task}] [{i+1}/{len(qa_data)}] Processing: {video_name}")
        print(f"Prompt: {prompt}")
        print(f"{'='*50}")

        try:
            response = analyze_video(video_path, prompt)
            print(f"\n--- {MODEL_NAME} Response ---")
            print(response)
        except Exception as e:
            response = f"Error: {e}"
            print(f"Error processing {video_name}: {e}")

        # Extract predicted (A/B/C) and show as Left/Middle/Right
        gt = (item.get("answer") or "").strip().upper()
        predicted = extract_predicted(response)
        if predicted is not None:
            extracted_position = LETTER_TO_POSITION.get(predicted, predicted)
            print(f"Extracted: {extracted_position} ({predicted})")
        else:
            extracted_position = "(none)"
            print(f"Extracted: (none)")

        if gt in "ABC":
            gt_position = LETTER_TO_POSITION.get(gt, gt)
            print(f"Ground truth: {gt_position} ({gt})")
            match = predicted == gt
            if match:
                correct_so_far += 1
            print(f"Match: {'Yes' if match else 'No'}")
        else:
            print(f"Ground truth: {gt or '(none)'}")
            match = False

        total_so_far = i + 1
        acc_pct = (100.0 * correct_so_far / total_so_far) if total_so_far else 0
        print(f"Accuracy so far: {correct_so_far}/{total_so_far} ({acc_pct:.1f}%)")

        result = item.copy()
        result["question"] = prompt
        result["model"] = MODEL_NAME
        result["response"] = response
        results.append(result)

        save_results(results, output_path)
        print(f"Progress saved ({len(results)} items)")

    print(f"\n{'='*50}")
    print(f"[{task}] Results saved to: {output_path}")
    print(f"[{task}] Total processed: {len(results)} items")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]


[card] [1/50] Processing: card_001.mp4
Prompt: Track the Queen of Hearts and answer where is the Queen of Hearts at the end of the video.
--- Processing /content/card/card_001.mp4 ---


molmo-utils using decord by default to read video.



--- Molmo2-8B-finetuned Response ---
<tracks coords="0.0 1 500 504;0.5 1 500 504;1.0 1 500 504;1.5 1 500 431;2.0 1 571 306;2.5 1 664 293;3.0 1 735 389;3.5 1 750 504;4.0 1 750 504;4.5 1 750 504;5.0 1 750 504;5.5 1 750 504;6.0 1 750 504;6.5 1 750 504;7.0 1 750 504;7.5 1 750 480;8.0 1 704 330;8.5 1 614 280;9.0 1 532 352;9.5 1 500 493;10.0 1 539 339;10.5 1 630 280;11.0 1 710 341;11.5 1 750 504;12.0 1 750 504">the Queen of Hearts</tracks> Answer: right.<|im_end|>
Extracted: Right (C)
Ground truth: Right (C)
Match: Yes
Accuracy so far: 1/1 (100.0%)
Progress saved (1 items)

[card] [2/50] Processing: card_002.mp4
Prompt: Track the Queen of Hearts and answer where is the Queen of Hearts at the end of the video.
--- Processing /content/card/card_002.mp4 ---

--- Molmo2-8B-finetuned Response ---
<tracks coords="0.0 1 750 517;0.5 1 750 517;1.0 1 745 500;4.5 1 442 775;5.0 1 286 646;5.5 1 252 538;6.0 1 355 738;6.5 1 539 788;7.0 1 705 662;7.5 1 750 500;8.0 1 750 500;8.5 1 750 500;9.0 1 750 500;9.5 